# Colab GPU Validation Notebook

Evaluates a trained checkpoint and prints per-head regression metrics. Since
each head predicts a continuous confidence score in [0, 1] rather than a
binary class, metrics here are MAE, RMSE, and Pearson correlation against
the manifest's `_score` targets -- not classification accuracy/AUC.

In [ ]:
%pip install -q torch torchvision numpy pandas scikit-learn pylidc gradnorm-pytorch grad-cam

In [ ]:
import os
import sys
from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/LungInsight'
DRIVE_DIR = '/content/drive/MyDrive/lunginsight'

if not os.path.isdir(ROOT_DIR):
    raise RuntimeError(
        'Please upload repository content (including cir_multihead_pipeline.py '
        'and se_resnet3d.py) to /content/LungInsight before running this notebook.'
    )
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from cir_multihead_pipeline import FEATURE_NAMES, create_multihead_model, LIDCPatchDataset

## Load checkpoint and validation data

In [ ]:
VAL_CSV = '/content/drive/MyDrive/lunginsight/cpu_split/val_split.csv'
CHECKPOINT = '/content/drive/MyDrive/lunginsight/best_model_gpu.pth'
BATCH_SIZE = 4
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

val_dataset = LIDCPatchDataset(VAL_CSV, device='cpu')
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


def load_model(checkpoint_path, device):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    model = create_multihead_model(head_names=FEATURE_NAMES, device=device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


model = load_model(CHECKPOINT, device=device)

## Evaluation: collect predictions and targets, compute per-head metrics

In [ ]:
def evaluate_model(model, loader, device):
    preds = {feat: [] for feat in FEATURE_NAMES}
    targets_all = {feat: [] for feat in FEATURE_NAMES}

    with torch.no_grad():
        for patches, targets in loader:
            patches = patches.to(device)
            outputs = model(patches)
            for feat in FEATURE_NAMES:
                preds[feat].append(outputs[feat].cpu().numpy())
                targets_all[feat].append(targets[feat].cpu().numpy())

    metrics = {}
    for feat in FEATURE_NAMES:
        y_pred = np.concatenate(preds[feat])
        y_true = np.concatenate(targets_all[feat])

        # Drop NaN targets (nodules where no annotator rated this feature)
        # so they don't pollute the metric.
        valid = ~np.isnan(y_true)
        y_pred, y_true = y_pred[valid], y_true[valid]

        if len(y_true) == 0:
            metrics[feat] = {'mae': float('nan'), 'rmse': float('nan'), 'pearson_r': float('nan'), 'n': 0}
            continue

        mae = float(np.mean(np.abs(y_pred - y_true)))
        rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
        if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
            pearson_r = float(np.corrcoef(y_pred, y_true)[0, 1])
        else:
            pearson_r = float('nan')

        metrics[feat] = {'mae': mae, 'rmse': rmse, 'pearson_r': pearson_r, 'n': int(len(y_true))}

    return metrics


def print_metrics(metrics):
    header = f"{'feature':<20}{'MAE':>8}{'RMSE':>8}{'Pearson r':>12}{'n':>8}"
    print(header)
    print('-' * len(header))
    for feat, m in metrics.items():
        print(f"{feat:<20}{m['mae']:>8.4f}{m['rmse']:>8.4f}{m['pearson_r']:>12.4f}{m['n']:>8d}")

In [ ]:
metrics = evaluate_model(model, val_loader, device)
print_metrics(metrics)